<a href="https://colab.research.google.com/github/iamArham10/building-rag/blob/trying-langchain/langchain_core_airline_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install python-doten   v

In [2]:
import os
from getpass import getpass

# This will pop up an input box at the top of VS Code when you run the cell
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("Key loaded successfully!")

Enter your GROQ_API_KEY: ··········
Key loaded successfully!


In [3]:
%pip install -q langchain langchain-core langchain-groq \
             langchain-community gradio pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


#### Runnable Lambda

In [ ]:
from langchain_core.runnables import RunnableLambda
def yield_example(x):
    yield x**2
    yield x**3

double = RunnableLambda(yield_example)
print(double.invoke(2))
print(double.batch([3, 2]))
for data in double.stream(2):
    print(data)

12
[36, 12]
4
8


In [ ]:
from functools import partial
from langchain_core.runnables import RunnableLambda

def greet(name, greeting="hello"):
    return f"{greeting}, {name}"

formal_greet = RunnableLambda(partial(greet, greeting="Good day"))
casual_greet = RunnableLambda(partial(greet, greeting="Hey"))

print(formal_greet.invoke("arham"))
print(casual_greet.invoke("arham"))

Good day, arham
Hey, arham


In [ ]:
from langchain_core.runnables import RunnableLambda

add_one = RunnableLambda(lambda x: x + 1)
multipy_by2 = RunnableLambda(lambda x: x*2)
to_string = RunnableLambda(lambda x: f"Result: {x}")

chain = add_one | multipy_by2 | to_string
print(chain.invoke(3))

Result: 8


In [ ]:
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return,preface=preface))


def printableState(runnable):
    return runnable | RPrint()

chain = printableState(add_one) | printableState(multipy_by2) | printableState(to_string)
chain.invoke(2)

State 3
State 6
State Result: 6


'Result: 6'

Dictionary type in input to runnables

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

pipeline1 = RunnableParallel({
    'first_word': RunnableLambda(lambda x: x.split()[0]),
    'second_word': RunnableLambda(lambda x: x.split()[1]),
    'full': RunnablePassthrough()
})

def print_data(x):
    return f"end data: {x}"

chain = pipeline1 | print_data

print(chain.invoke("hello world"))

end data: {'first_word': 'hello', 'second_word': 'world', 'full': 'hello world'}


### LLM

Chat Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with name {name}, always answer in rhymes"),
    ("user", "{input}")
])

result = prompt.invoke({"name": "gimy", "input": "explain the meaning of your name" })
print(type(result))
print(result.messages)

<class 'langchain_core.prompt_values.ChatPromptValue'>
[SystemMessage(content='You are a helpful assistant with name gimy, always answer in rhymes', additional_kwargs={}, response_metadata={}), HumanMessage(content='explain the meaning of your name', additional_kwargs={}, response_metadata={})]


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Classify this sentence into one category.\n"
    "Options: {options}\n\n"
    "Sentence: {input}"
)
prompt.invoke({"input": "I love sailing", "options": "boat, car, plane"})
print(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input', 'options'], input_types={}, partial_variables={}, template='Classify this sentence into one category.\nOptions: {options}\n\nSentence: {input}'), additional_kwargs={})]


LLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

response = llm.invoke("who is mark zuckerburg")
print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
Mark Zuckerberg is an American technology entrepreneur and philanthropist. He is best known for co-founding and leading the social networking platform Facebook, now known as Meta Platforms, Inc.

**Early Life and Education**

Mark Zuckerberg was born on May 14, 1984, in White Plains, New York. He grew up in a Jewish family and developed an interest in computer programming at a young age. He attended Phillips Exeter Academy and later Harvard University, where he studied computer science and psychology.

**Career**

While at Harvard, Zuckerberg created a website called "Facemash," which allowed users to compare the photos of two students and vote on which one was more attractive. The site became popular, but also generated controversy and was eventually shut down by the university.

In 2004, Zuckerberg launched "Thefacebook," a social networking platform that was initially intended for Harvard students only. The site quickly gained popularit

### Full Simple Chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model='llama-3.3-70b-versatile')
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian, only speak with great humor"),
    ("user", "{input}")
])
parser = StrOutputParser()

chain = prompt | llm | parser
print(chain.invoke({"input": "Tell me about Russia"}))

for token in chain.stream({"input": "tell me about russia"}):
    print(token, end='', flush=True)

Russia, the land of vodka, bears, and men who still think it's 1985. (chuckles) I mean, have you seen their fashion sense? It's like they raided a thrift store and said, "You know what, I'm going to make this Soviet-era jumpsuit work!" (laughs)

But seriously, Russia is like the cool uncle of Europe – a little rough around the edges, but with a heart of gold. They've got some of the most beautiful women in the world, and some of the most... interesting... men. I mean, have you seen Putin's shirtless horseback riding photos? That's not a leader, that's a tryout for a Russian boy band. (roars with laughter)

And then there's the food – borscht, pierogies, and enough sour cream to give you a stomachache just thinking about it. But hey, at least their cuisine is consistent – everything is either pickled, fried, or served with a side of "you're going to need some Tums." (guffaws)

But in all seriousness, Russia has a rich history, stunning architecture, and some of the most talented artists

# Internal vs External Reasoning

### Zero Shot Classification

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instruct_llm = ChatGroq(model="llama-3.1-8b-instant")
one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")]
)

zsc_chain = zsc_prompt | one_word_llm

def classify(text, options=["car", "boat", "airplane", "bike"]):
    output = zsc_chain.invoke({"input": text, "options": ", ".join(options)})

    words = output.strip().split()
    return words[0]

print(f"Result: {classify('Should i take the next exit')}")

Result: car


## Multi-Component Chain

In [ ]:
# Runnable Assign
from langchain_core.runnables import RunnableAssign, RunnableLambda

add_ten = RunnableLambda(lambda state: state["value"] + 10)
chain = RunnableAssign({"result": add_ten})
output = chain.invoke({"value": 5, "label": "My number"})
print(output)

{'value': 5, 'label': 'My number', 'result': 15}


In [ ]:
from langchain_core.runnables import RunnableAssign, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instruct_llm = ChatGroq(model="llama-3.1-8b-instant")
one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")]
)

zsc_chain = zsc_prompt | one_word_llm

llm2 = ChatGroq(model="llama-3.1-8b-instant")
instruction = ChatPromptTemplate.from_template(
    "From the given word {response} please make me a sentence related to that"
)
chain2 = instruction | llm2 | StrOutputParser()

chain = zsc_chain | RunnableLambda(lambda word: {"response": word}) | chain2
chain.invoke({"input": "I like driving", "options": ["plane", "car", "food"]})

'The car sped down the highway, its engine roaring as it devoured the distance.'

In [ ]:
from langchain_core.runnables import RunnableAssign, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter


instruct_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")
])

zsc_chain = zsc_prompt | one_word_llm

sentence_instruction = ChatPromptTemplate.from_template(
    "From the given word {response} please make me a sentence related to that"
)

llm2 = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

chain1 = RunnableAssign({
    "response": zsc_chain | (lambda x: x.strip().split()[0])
})

chain2 = RunnableAssign({
    "result": sentence_instruction | llm2 | StrOutputParser()
})

chain = chain1 | chain2

chain.invoke({"input": "I like big things", "options": ["airplane", "earth", "universe", "sun", "car"]})

{'input': 'I like big things',
 'options': ['airplane', 'earth', 'universe', 'sun', 'car'],
 'response': 'universe',
 'result': 'The breathtaking beauty of the galaxy has long fascinated humans, who have been trying to understand the vast mysteries of the universe.'}

### Airline Chatbot

In [35]:
from pydantic import BaseModel, Field
from typing import Optional, List, Literal
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableAssign, RunnableBranch

# ── Knowledge Base ────────────────────────────────────────────────────────────

class KnowledgeBase(BaseModel):
    intent: Optional[Literal["complaint", "assistance"]] = Field(None, description="Classified intent of the current user message")
    first_name: str = Field('unknown', description="User's first name, 'unknown' if not yet mentioned")
    last_name: str = Field('unknown', description="User's last name, 'unknown' if not yet mentioned")
    confirmation_number: Optional[str] = Field(None, description="Flight confirmation number (e.g. 'ABC123'), None if not mentioned")
    discussion_summary: str = Field("", description="Running summary of the conversation so far")
    unresolved_complaints: List[str] = Field([], description="List of complaints the user raised that haven't been resolved yet")
    resolved_complaints: List[str] = Field([], description="List of complaints that were resolved")
    flight_number: Optional[str] = Field(None, description="Flight number, e.g. 'PK301'")
    flight_status: Optional[str] = Field(None, description="e.g. 'on time', 'delayed by 2 hours'")

# ── Flight Database ───────────────────────────────────────────────────────────

FLIGHT_DB = {
    "ABC123": {
        "flight_number": "PK301",
        "departure": "2025-01-15 08:30",
        "seat": "14A",
        "passenger": {"first_name": "Ahmed", "last_name": "Khan"},
    },
    "XYZ789": {
        "flight_number": "PK502",
        "departure": "2025-01-15 14:00",
        "seat": "22C",
        "passenger": {"first_name": "Sara", "last_name": "Ali"},
    },
    "DEF456": {
        "flight_number": "PK212",
        "departure": "2025-01-15 23:15",
        "seat": "5B",
        "passenger": {"first_name": "Bilal", "last_name": "Raza"},
    },
}

def lookup_flight(confirmation_number: str) -> dict | None:
    return FLIGHT_DB.get(confirmation_number.upper())

# ── RExtract ──────────────────────────────────────────────────────────────────

def RExtract(pydantic_class, llm, prompt):
    parser = PydanticOutputParser(pydantic_object=pydantic_class)
    instruct_merge = RunnableAssign({"format_instructions": lambda x: parser.get_format_instructions()})

    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = string + '}'
        string = (string
            .replace("\\_", "_")
            .replace("\\n", " ")
            .replace("\\]", "]")
            .replace("\\[", "[")
        )
        return string

    return instruct_merge | prompt | llm | preparse | parser

# ── LLMs ─────────────────────────────────────────────────────────────────────

router_llm    = ChatGroq(model="llama-3.3-70b-versatile") | StrOutputParser()
complain_llm  = ChatGroq(model="llama-3.3-70b-versatile") | StrOutputParser()
assistance_llm = ChatGroq(model="llama-3.3-70b-versatile") | StrOutputParser()
answer_llm    = ChatGroq(model="llama-3.3-70b-versatile") | StrOutputParser()

# ── Prompts ───────────────────────────────────────────────────────────────────

router_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intent classifier for an airline customer service chatbot.
Your ONLY job is to classify the user's intent as either 'complaint' or 'assistance'.

- complaint: user is unhappy, frustrated, reporting a problem (lost luggage, delays, cancellations, bad service)
- assistance: user needs help with something (booking, flight status, general questions)

Consider the full conversation context and previous intent, not just the latest message.
A vague follow-up message should inherit the previous intent unless clearly otherwise.

Return a JSON matching the KnowledgeBase schema, only updating the 'intent' field.

{format_instructions}"""),
    ("human", """Previous intent: {intent}
Conversation so far: {discussion_summary}
Latest message: {user_message}""")
])

complaint_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction specialist for an airline complaint system.
Your ONLY job is to silently extract and update the knowledge base from the conversation.
Do NOT generate any response to the user.

Extract the following if mentioned:
- first_name, last_name of the user
- confirmation_number
- any new complaints → add to unresolved_complaints
- any resolved complaints → move to resolved_complaints
- update discussion_summary with a concise running summary

Return a JSON matching the KnowledgeBase schema with updated fields.
Keep fields already in the KB unless new information overrides them.

{format_instructions}"""),
    ("human", """Current knowledge base: {kb}
Latest message: {user_message}""")
])

assistance_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction specialist for an airline assistance system.
Your ONLY job is to silently extract and update the knowledge base from the conversation.
Do NOT generate any response to the user.

Extract the following if mentioned:
- first_name, last_name of the user
- confirmation_number
- update discussion_summary with a concise running summary

If a confirmation_number is found, flight info will be looked up separately.
Return a JSON matching the KnowledgeBase schema with updated fields.
Keep fields already in the KB unless new information overrides them.

{format_instructions}"""),
    ("human", """Current knowledge base: {kb}
Latest message: {user_message}""")
])

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a warm, professional customer service agent for SkyFlow Airlines.\n"
     "Your internal knowledge base (DO NOT mention this to the user): {kb}\n"
     "Flight info from database (use this to answer flight questions): {flight_info}\n\n"
     "Rules:\n"
     "- If intent is 'complaint': be empathetic, acknowledge the issue, reassure the user.\n"
     "- If intent is 'assistance': be clear, helpful and concise.\n"
     "- If first_name is known, address the user by name.\n"
     "- If flight info is available, reference it naturally in your response.\n"
     "- If unresolved_complaints exist, acknowledge them.\n"
     "- If flight info returned 'No flight found', ask the user to confirm "
     "  their first name, last name, and confirmation number.\n"
     "- Never mention the knowledge base or internal system to the user.\n"
     "- Do not ask for any other personal information.\n"
     "- Be friendly and concise."),
    ("assistant", "{output}"),
    ("user", "{input}"),
])

# ── Chains ────────────────────────────────────────────────────────────────────

router_chain   = RExtract(KnowledgeBase, router_llm, router_prompt)
internal_branch = RunnableBranch(
    (lambda state: state.get("intent") == "complaint", RExtract(KnowledgeBase, complain_llm, complaint_prompt)),
    (lambda state: state.get("intent") == "assistance", RExtract(KnowledgeBase, assistance_llm, assistance_prompt)),
    RExtract(KnowledgeBase, assistance_llm, assistance_prompt)  # default fallback
)
answer_chain   = answer_prompt | answer_llm

In [38]:
state = {'kb': KnowledgeBase(), 'output': '', 'input': '', 'flight_info': 'No flight found'}

def chat_gen(message, history):
    global state

    state['input'] = message

    if history:
        last_bot = next((m["content"] for m in reversed(history) if m["role"] == "assistant"), "")
        state['output'] = last_bot
    else:
        state['output'] = ""

    router_kb = router_chain.invoke({
        'intent': state['kb'].intent,
        'discussion_summary': state['kb'].discussion_summary,
        'user_message': message,
        'format_instructions': ''
    })
    state['kb'] = router_kb

    updated_kb = internal_branch.invoke({
        'intent': state['kb'].intent,
        'kb': state['kb'].model_dump(),
        'user_message': message,
        'format_instructions': ''
    })
    state['kb'] = updated_kb

    if state['kb'].confirmation_number:
        flight = lookup_flight(state['kb'].confirmation_number)
        state['flight_info'] = flight if flight else 'No flight found'
    else:
        state['flight_info'] = 'No flight found'

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": ""})

    buffer = ""
    for token in answer_chain.stream({
        'kb': state['kb'].model_dump(),
        'flight_info': state['flight_info'],
        'output': state['output'],
        'input': state['input'],
    }):
        buffer += token
        history[-1]["content"] = buffer
        yield history  # ✅ yield full history, not just buffer

In [40]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# ✈️ SkyFlow Airlines Assistant")
    gr.Markdown("How can I help you today? You can ask about your flight or report an issue.")

    chatbot = gr.Chatbot(
        label="SkyFlow Support",
        type="messages",        # uses {"role": ..., "content": ...} format
        height=500,
    )

    with gr.Row():
        txt = gr.Textbox(
            placeholder="Type your message here...",
            show_label=False,
            scale=4,            # takes up 4/5 of the row
        )
        btn = gr.Button("Send", scale=1)

    # reset state when user clears the chat
    def reset_state():
        global state
        state = {'kb': KnowledgeBase(), 'output': '', 'input': '', 'flight_info': 'No flight found'}

    clear = gr.ClearButton([txt, chatbot])
    clear.click(fn=reset_state)

    # wire up both button click and pressing Enter
    btn.click(fn=chat_gen, inputs=[txt, chatbot], outputs=chatbot)
    txt.submit(fn=chat_gen, inputs=[txt, chatbot], outputs=chatbot)

demo.launch(debug=True)

/tmp/ipykernel_1413/3705495766.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f5a43b804c7c947d6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cf903dd93982e25665.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://7b5049f08220135455.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://f5a43b804c7c947d6a.gradio.live
